<img src="images/07.png" width="40%">

<img src="images/08.png" width="40%">

<img src="images/09.png" width="40%">

<img src="images/10.png" width="40%">

<img src="images/11.png" width="40%">

<img src="images/12.png" width="40%">

In [6]:
import torch

# ====================== 1. 生成人造数据集（对应笔记：y = X @ w_true + b_true + ε噪声） ======================
# 100个样本，1维特征
n_samples = 100
X = torch.randn(n_samples, 1)   # shape: [100,1]

# 真实参数（我们模型要去拟合的目标）
true_w = torch.tensor([[2.0]])
true_b = torch.tensor([3.0])

# 加高斯噪声 ε ~ N(0,0.3)
y = X @ true_w + true_b + torch.randn(n_samples, 1) * 0.3

# ====================== 2. 初始化待学习参数 w,b ======================
w = torch.randn(1, 1, requires_grad=True) # 权重，开启梯度追踪
b = torch.zeros(1, 1, requires_grad=True) # 偏置，开启梯度追踪

learning_rate = 0.1
epochs = 200

# ====================== 3. 核心训练循环：前向→损失→反向求梯度→更新参数→梯度清零 ======================
for epoch in range(epochs):
    # 前向传播：预测 y_hat = X@w + b
    y_pred = X @ w + b
    
    # 计算MSE损失（均方误差）
    loss = torch.mean((y_pred - y) ** 2)

    # 反向传播：自动求导，计算 loss 对 w,b 的梯度，存入w.grad、b.grad
    loss.backward()

    # 参数更新：不要计算梯度！用torch.no_grad()关闭计算图
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 梯度清零！非常关键，否则梯度会不断累加
    w.grad.zero_()
    b.grad.zero_()
    
    # 每20轮打印一次训练信息
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, w: {w.item():.3f}, b: {b.item():.3f}")

print("\n====训练完成====")
print(f"学习得到 w = {w.item():.3f}, b = {b.item():.3f}")
print(f"真实参数 w = {true_w.item()}, b = {true_b.item()}")


Epoch [20/200], Loss: 0.1014, w: 2.011, b: 2.980
Epoch [40/200], Loss: 0.1002, w: 2.019, b: 3.005
Epoch [60/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [80/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [100/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [120/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [140/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [160/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [180/200], Loss: 0.1002, w: 2.018, b: 3.005
Epoch [200/200], Loss: 0.1002, w: 2.018, b: 3.005

====训练完成====
学习得到 w = 2.018, b = 3.005
真实参数 w = 2.0, b = 3.0


<img src="images/13.png" width="70%">

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 生成模拟数据集（同上）
x = torch.randn(100, 1)
true_w = 2.0
true_b = 3.0
y = x * true_w + true_b + torch.randn(100, 1) * 0.3


# 2. 定义模型：继承 nn.Module
class LinearRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # 线性层：y = x*w^T + b，自动封装了w和b参数
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        # 前向传播逻辑
        return self.linear(x)


# 3. 初始化模型、损失函数、优化器
model = LinearRegression(input_dim=1)
criterion = nn.MSELoss()  # 均方误差损失
optimizer = optim.SGD(model.parameters(), lr=0.03)  # SGD优化器

# 4. 训练循环（标准范式）
epochs = 100
for epoch in range(epochs):
    # ① 梯度清零（每轮必须做）
    optimizer.zero_grad()

    # ② 前向传播：输入x，得到预测值
    y_pred = model(x)

    # ③ 计算损失
    loss = criterion(y_pred, y)

    # ④ 反向传播：计算所有参数梯度
    loss.backward()

    # ⑤ 更新参数
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

# 5. 查看训练结果
w = model.linear.weight.item()
b = model.linear.bias.item()
print(f"\n训练后的权重w: {w:.4f}，偏置b: {b:.4f}")
print(f"真实权重w: {true_w:.4f}，偏置b: {true_b:.4f}")


Epoch 10, Loss: 4.1496
Epoch 20, Loss: 1.0882
Epoch 30, Loss: 0.3478
Epoch 40, Loss: 0.1679
Epoch 50, Loss: 0.1240
Epoch 60, Loss: 0.1131
Epoch 70, Loss: 0.1104
Epoch 80, Loss: 0.1098
Epoch 90, Loss: 0.1096
Epoch 100, Loss: 0.1095

训练后的权重w: 2.0132，偏置b: 2.9704
真实权重w: 2.0000，偏置b: 3.0000
